# Lesson 4: Bring an existing Brain-Score model

**You will learn:** how models registered for classic Brain-Score (vision and language) become subjects, and how to translate one into a `BrainScoreModel`.
**Requires:** CPU only, no downloads, under a minute.

Classic Brain-Score vision models are `BrainModel` objects with methods such as `look_at` and `start_recording`. Language models are `ArtificialSubject` objects with `digest_text`. You do not need to rewrite them. `brainscore.load_model(...)` wraps them in an **adapter** that presents them as a subject. Old benchmarks also still run on any subject.

In [1]:
import warnings
warnings.filterwarnings('ignore')  # keep the lesson output short

In [2]:
import tempfile, pathlib
import numpy as np
from PIL import Image
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# Eight small synthetic images: four bright, four dark. They stand in for real stimuli.
folder = pathlib.Path(tempfile.mkdtemp())
rng = np.random.default_rng(0)
rows = []
for i in range(8):
    bright = i % 2 == 0
    low, high = (150, 255) if bright else (0, 100)
    path = folder / f'image_{i}.png'
    Image.fromarray(rng.integers(low, high, (16, 16, 3), dtype=np.uint8)).save(path)
    rows.append({'stimulus_id': f'image_{i}', 'image_path': str(path),
                 'label': 'bright' if bright else 'dark'})

stimuli = StimulusSet(rows)
stimuli.identifier = 'toy-bright-dark'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli

,stimulus_id,image_path,label
0,image_0,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
1,image_1,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
2,image_2,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
3,image_3,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
4,image_4,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
5,image_5,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
6,image_6,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
7,image_7,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark


In [3]:
import torch

torch.manual_seed(0)
# A tiny, untrained convolutional network. Its weights are random.
net = torch.nn.Sequential(
    torch.nn.Conv2d(3, 4, kernel_size=3),  # layer '0'
    torch.nn.ReLU(),                       # layer '1'
    torch.nn.AdaptiveAvgPool2d(2),         # layer '2'
    torch.nn.Flatten(),                    # layer '3'
    torch.nn.Linear(16, 8),                # layer '4'
).eval()

def preprocess(paths):
    """Load image files into a (batch, 3, height, width) float array."""
    images = [np.asarray(Image.open(p).convert('RGB'), dtype=np.float32) / 255 for p in paths]
    return np.stack(images).transpose(0, 3, 1, 2)

## A classic vision registration

This is how a vision model plugin is usually written: a `PytorchWrapper` for activations, inside a `ModelCommitment` with a region-to-layer map.

In [4]:
from brainscore_vision.model_helpers.activations.pytorch import PytorchWrapper
from brainscore_vision.model_helpers.brain_transformation import ModelCommitment

legacy_model = ModelCommitment(
    identifier='toy-cnn-classic',
    activations_model=PytorchWrapper(identifier='toy-cnn-classic', model=net, preprocessing=preprocess),
    layers=['0', '4'],
    region_layer_map={'IT': '4'},
)

## The adapter makes it a subject

`load_model` does this step for you when it finds a registered vision model. Here it is done by hand so you can see it.

In [5]:
from brainscore_vision.compat.unified_adapter import VisionModelAdapter
from brainscore_core.streaming import BufferedSession

subject = VisionModelAdapter(legacy_model)
print('outputs:', subject.out_channels)

session = BufferedSession(stimulus_set=stimuli, requested_output_channels=['neural:IT'], time_bins=[(70, 170)])
subject.participate(session)
legacy_activity = session.collect()
legacy_activity.shape

outputs: {'behavior', 'neural:IT'}


activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

(8, 8)

## Translating the registration

If you want the new features (several input types, perturbations, closed-loop control), translate the registration. The pieces map one to one:

| Classic | `BrainScoreModel` |
| --- | --- |
| `PytorchWrapper(..., preprocessing=...)` | `VisionWrapper(..., preprocessing=...)` in `extractor={'vision': ...}` |
| `region_layer_map={'IT': '4'}` | `region_layer_map={'IT': '4'}` |
| `behavioral_readout_layer='4'` | `behavioral_readout_layer='4'` |
| `look_at(stimuli)` | `participate(session)` (or `process(stimuli)` interactively) |

In [6]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper

new_model = BrainScoreModel(
    identifier='toy-cnn-new',
    model=net,
    extractor={'vision': VisionWrapper(identifier='toy-cnn-new', model=net, preprocessing=preprocess)},
    region_layer_map={'IT': '4'},
)
session = BufferedSession(stimulus_set=stimuli, requested_output_channels=['neural:IT'])
new_model.participate(session)
new_activity = session.collect()

np.allclose(np.sort(new_activity.values, axis=None), np.sort(legacy_activity.values, axis=None))

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

True

Both versions record the same activity from the same network.

## Loading by name

For registered models you do not build anything:

```python
import brainscore
model = brainscore.load_model('alexnet')          # classic vision plugin, adapted automatically
score = brainscore.score('alexnet', 'MajajHong2015.IT-pls')   # classic benchmark
```

(Not run here because it downloads weights and data.)

## Summary

- Classic vision and language models become subjects through **adapters**; `load_model` applies them automatically. They are permanent, not a migration step.
- Classic benchmarks run on any subject.
- Translating to `BrainScoreModel` is a mechanical renaming of the same parts.

**Next:** [Lesson 5](05_layers_and_regions.ipynb) looks more closely at layers and regions.